In [ ]:


print("     INTERNSHIP SUPPORT CHATBOT")
print("AI-powered chatbot for answering intern queries")

!pip install -q pandas numpy scikit-learn nltk joblib gradio

import pandas as pd
import numpy as np
import re
import string
import joblib
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk
from nltk.corpus import stopwords

import gradio as gr

nltk.download('stopwords')

os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("app", exist_ok=True)

print("Project folders created successfully.")

faq_data = [

    # Internship report
    {
        "question": "How do I submit my internship report?",
        "answer": "You should submit your internship report through the internship portal before the stated deadline.",
        "category": "report"
    },
    {
        "question": "Where can I upload my internship report?",
        "answer": "You can upload your internship report through the internship portal under the report submission section.",
        "category": "report"
    },
    {
        "question": "What should be included in the internship report?",
        "answer": "Your report should normally include an introduction, company information, tasks completed, skills learned, challenges, and conclusion.",
        "category": "report"
    },

    # Supervisor
    {
        "question": "Who is my internship supervisor?",
        "answer": "Your internship supervisor is the person assigned by your organization or university. Please check your internship assignment details.",
        "category": "supervisor"
    },
    {
        "question": "How can I contact my supervisor?",
        "answer": "You can contact your assigned supervisor using the email or contact information provided in your internship details.",
        "category": "supervisor"
    },

    # Working hours
    {
        "question": "How many hours do interns work?",
        "answer": "Internship working hours depend on the organization. Please follow the working schedule provided by your internship coordinator.",
        "category": "working_hours"
    },
    {
        "question": "What are the internship working hours?",
        "answer": "Your working hours are determined by your organization and should be mentioned in your internship schedule.",
        "category": "working_hours"
    },

    # Attendance
    {
        "question": "What happens if I am absent?",
        "answer": "If you need to be absent, inform your supervisor as soon as possible and follow the organization's absence policy.",
        "category": "attendance"
    },
    {
        "question": "How do I report an absence?",
        "answer": "Contact your internship supervisor or coordinator and explain the reason for your absence.",
        "category": "attendance"
    },

    # Leave
    {
        "question": "Can interns take leave?",
        "answer": "Intern leave depends on the organization's policy. You should request leave from your supervisor in advance whenever possible.",
        "category": "leave"
    },

    # Certificate
    {
        "question": "How can I get my internship certificate?",
        "answer": "After completing your internship requirements, contact your supervisor or internship coordinator to request your completion certificate.",
        "category": "certificate"
    },
    {
        "question": "When will I receive my internship certificate?",
        "answer": "The certificate is normally provided after successful completion of the internship and required documentation.",
        "category": "certificate"
    },

    # Evaluation
    {
        "question": "How will my internship performance be evaluated?",
        "answer": "Your performance may be evaluated based on attendance, assigned tasks, supervisor feedback, reports, and overall participation.",
        "category": "evaluation"
    },

    # Documents
    {
        "question": "What documents are required for the internship?",
        "answer": "Required documents may include your CV, identification documents, internship offer letter, university forms, and other documents requested by the organization.",
        "category": "documents"
    },

    # HR
    {
        "question": "How can I contact HR?",
        "answer": "You can contact the HR department using the official contact details provided by your organization.",
        "category": "hr"
    },

    # Stipend
    {
        "question": "Do interns receive a stipend?",
        "answer": "Whether an internship provides a stipend depends on the organization's internship policy and offer letter.",
        "category": "stipend"
    },

    # Tasks
    {
        "question": "What should I do if I do not understand my task?",
        "answer": "Ask your supervisor or assigned mentor for clarification. It is better to ask questions than to complete a task incorrectly.",
        "category": "tasks"
    },

    # Training
    {
        "question": "Is internship training provided?",
        "answer": "Training depends on the organization. Your supervisor or HR department can provide information about available training.",
        "category": "training"
    },

    # Internship extension
    {
        "question": "Can my internship be extended?",
        "answer": "An internship may be extended if both the intern and organization agree and the relevant policy allows it.",
        "category": "extension"
    },

    # Resignation
    {
        "question": "Can I leave my internship early?",
        "answer": "If you need to leave your internship early, discuss the situation with your supervisor and follow the organization's termination or withdrawal procedure.",
        "category": "termination"
    },

    # General
    {
        "question": "What is an internship?",
        "answer": "An internship is a temporary work or training experience that allows students or graduates to gain practical knowledge and professional skills.",
        "category": "general"
    }
]

df = pd.DataFrame(faq_data)

print("Dataset created successfully!")
print("Number of FAQ records:", len(df))

df.head()

df.to_csv("data/internship_faq.csv", index=False)

print("FAQ dataset saved to:")
print("data/internship_faq.csv")

print(df.shape)
print()

print(df["category"].value_counts())

stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    text = text.lower()

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Split into words
    words = text.split()

    # Remove stopwords
    words = [word for word in words if word not in stop_words]

    return " ".join(words)

test_text = "How do I submit my internship report?"

print("Original:")
print(test_text)

print("\nProcessed:")
print(preprocess_text(test_text))

df["processed_question"] = df["question"].apply(preprocess_text)

df[["question", "processed_question"]].head(10)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=5000
)

X = vectorizer.fit_transform(df["processed_question"])

print("TF-IDF matrix shape:", X.shape)

def chatbot_response(user_question, threshold=0.20):

    if not user_question or user_question.strip() == "":
        return "Please enter a question."

    # Preprocess user question
    processed_question = preprocess_text(user_question)

    # Convert user question to TF-IDF
    user_vector = vectorizer.transform([processed_question])

    # Calculate similarity
    similarities = cosine_similarity(user_vector, X)[0]

    # Find best match
    best_index = np.argmax(similarities)
    best_score = similarities[best_index]

    # Get answer
    if best_score >= threshold:
        answer = df.iloc[best_index]["answer"]
        category = df.iloc[best_index]["category"]

        return (
            f"{answer}\n\n"
            f"Category: {category}\n"
            f"Confidence: {best_score:.2f}"
        )

    else:
        return (
            "I'm sorry, I don't have enough information to answer "
            "that question. Please contact your internship supervisor "
            "or HR department."
        )

questions = [
    "How can I submit my report?",
    "Who is my supervisor?",
    "What happens if I miss work?",
    "Can interns get leave?",
    "How do I get my certificate?"
]

for question in questions:
    print("USER:", question)
    print("BOT:", chatbot_response(question))
    print("-" * 70)

user_question = input("Ask your internship question: ")

print("\nChatbot:")
print(chatbot_response(user_question))

additional_data = [

    {
        "question": "Where do I submit my internship report?",
        "answer": "You should submit your internship report through the internship portal before the stated deadline.",
        "category": "report"
    },

    {
        "question": "How can I upload my report?",
        "answer": "You can upload your internship report through the internship portal under the report submission section.",
        "category": "report"
    },

    {
        "question": "I need help with my internship supervisor",
        "answer": "Your internship supervisor is the person assigned by your organization or university. Please check your internship assignment details.",
        "category": "supervisor"
    },

    {
        "question": "What should I do when I miss internship work?",
        "answer": "If you need to be absent, inform your supervisor as soon as possible and follow the organization's absence policy.",
        "category": "attendance"
    },

    {
        "question": "Do I get paid during my internship?",
        "answer": "Whether an internship provides a stipend depends on the organization's internship policy and offer letter.",
        "category": "stipend"
    }
]

additional_df = pd.DataFrame(additional_data)

df = pd.concat([df, additional_df], ignore_index=True)

print("Updated dataset size:", len(df))

df["processed_question"] = df["question"].apply(preprocess_text)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=5000
)

X = vectorizer.fit_transform(df["processed_question"])

print("Model updated successfully.")
print("New TF-IDF matrix:", X.shape)

joblib.dump(vectorizer, "models/tfidf_vectorizer.pkl")

joblib.dump(
    {
        "data": df,
        "matrix": X
    },
    "models/chatbot_model.pkl"
)

print("Model saved successfully!")

def final_chatbot(user_question):

    if not user_question or user_question.strip() == "":
        return "Please enter a question."

    processed_question = preprocess_text(user_question)

    user_vector = loaded_vectorizer.transform(
        [processed_question]
    )

    similarities = cosine_similarity(
        user_vector,
        loaded_X
    )[0]

    best_index = np.argmax(similarities)
    best_score = similarities[best_index]

    if best_score >= 0.20:

        answer = loaded_df.iloc[best_index]["answer"]

        return answer

    return (
        "I'm sorry, I don't have information about that. "
        "Please contact your internship supervisor or HR department."
    )

def chat_function(message, history):
    response = final_chatbot(message)
    return response


demo = gr.ChatInterface(
    fn=chat_function,
    title="🎓 Internship Support Chatbot",
    description=(
        "Ask questions about your internship, including reports, "
        "supervisors, attendance, certificates, working hours, "
        "documents, HR, and other internship-related topics."
    ),
    textbox=gr.Textbox(
        placeholder="Ask your internship question...",
        container=True
    )
)

demo.launch()

def final_chatbot(user_question):

    if not user_question or user_question.strip() == "":
        return "Please enter a question."

    question_lower = user_question.lower().strip()

    greetings = [
        "hello",
        "hi",
        "hey",
        "good morning",
        "good afternoon",
        "good evening"
    ]

    if question_lower in greetings:
        return (
            "Hello! 👋 I'm the Internship Support Chatbot. "
            "How can I help you today?"
        )

    if question_lower in ["thanks", "thank you", "thank"]:
        return (
            "You're welcome! 😊 I'm happy to help."
        )

    processed_question = preprocess_text(user_question)

    user_vector = loaded_vectorizer.transform(
        [processed_question]
    )

    similarities = cosine_similarity(
        user_vector,
        loaded_X
    )[0]

    best_index = np.argmax(similarities)
    best_score = similarities[best_index]

    if best_score >= 0.20:
        return loaded_df.iloc[best_index]["answer"]

    return (
        "I'm sorry, I don't have information about that. "
        "Please contact your internship supervisor or HR department."
    )

import gradio as gr

print("Gradio version:", gr.__version__)

# ============================================================
# PROFESSIONAL INTERNSHIP SUPPORT CHATBOT
# LIGHT BLUE + BLACK CORPORATE DESIGN
# ============================================================

custom_css = """

/* =========================================================
   GLOBAL
   ========================================================= */

body {
    background: #f4f9fc !important;
    font-family: "Inter", "Segoe UI", Arial, sans-serif !important;
}

.gradio-container {
    max-width: 1400px !important;
    margin: auto !important;
    padding: 20px !important;
}


/* =========================================================
   HEADER
   ========================================================= */

.header {
    background: linear-gradient(
        135deg,
        #dff3ff 0%,
        #ffffff 100%
    );

    border: 1px solid #c7e7f8;

    border-radius: 20px;

    padding: 30px 35px;

    margin-bottom: 22px;

    box-shadow:
        0 8px 25px rgba(20, 80, 110, 0.08);
}

.header h1 {
    margin: 0;

    color: #101820;

    font-size: 32px;

    font-weight: 750;

    letter-spacing: -0.5px;
}

.header p {
    margin: 9px 0 15px 0;

    color: #52636f;

    font-size: 15px;
}

.online {
    display: inline-block;

    background: #e8f8ef;

    color: #16804b;

    border: 1px solid #bce8ce;

    padding: 7px 14px;

    border-radius: 20px;

    font-size: 13px;

    font-weight: 600;
}


/* =========================================================
   SIDEBAR
   ========================================================= */

.sidebar {
    background: #ffffff;

    border: 1px solid #d9eaf2;

    border-radius: 18px;

    padding: 22px;

    box-shadow:
        0 6px 20px rgba(20, 80, 110, 0.06);
}

.sidebar-title {
    color: #101820;

    font-size: 18px;

    font-weight: 700;

    margin-bottom: 5px;
}

.sidebar-subtitle {
    color: #71808a;

    font-size: 13px;

    line-height: 1.5;

    margin-bottom: 18px;
}

.category {
    background: #f4faff;

    border: 1px solid #d9edf7;

    border-radius: 12px;

    padding: 11px 13px;

    margin: 8px 0;

    color: #17232b;

    font-size: 13px;

    transition: 0.2s;
}

.category:hover {
    background: #e5f6ff;

    border-color: #8ed0ee;

    transform: translateX(3px);
}


/* =========================================================
   CHAT AREA
   ========================================================= */

.chat-area {
    background: #ffffff;

    border: 1px solid #d9eaf2;

    border-radius: 18px;

    padding: 15px;

    box-shadow:
        0 6px 20px rgba(20, 80, 110, 0.06);
}


/* =========================================================
   CHAT WINDOW
   ========================================================= */

.chatbot {
    border-radius: 15px !important;

    border: none !important;
}


/* =========================================================
   TEXT INPUT
   ========================================================= */

textarea {
    border: 1px solid #c9dfe9 !important;

    border-radius: 12px !important;

    background: #ffffff !important;

    color: #101820 !important;

    font-size: 14px !important;
}

textarea:focus {
    border-color: #55b7df !important;

    box-shadow:
        0 0 0 3px rgba(85, 183, 223, 0.15) !important;
}


/* =========================================================
   SEND BUTTON
   ========================================================= */

.primary-btn {
    background: #101820 !important;

    color: #ffffff !important;

    border: none !important;

    border-radius: 11px !important;

    font-weight: 650 !important;

    min-height: 45px !important;
}

.primary-btn:hover {
    background: #243641 !important;
}


/* =========================================================
   CLEAR BUTTON
   ========================================================= */

.secondary-btn {
    background: #e8f7fd !important;

    color: #101820 !important;

    border: 1px solid #b9dfef !important;

    border-radius: 11px !important;

    font-weight: 600 !important;

    min-height: 45px !important;
}

.secondary-btn:hover {
    background: #d8f1fc !important;
}


/* =========================================================
   EXAMPLE QUESTIONS
   ========================================================= */

.examples button {
    border-radius: 10px !important;

    border: 1px solid #d4e9f2 !important;

    background: #f7fcff !important;

    color: #24343d !important;
}


/* =========================================================
   FOOTER
   ========================================================= */

.footer {
    text-align: center;

    color: #7a8992;

    font-size: 12px;

    line-height: 1.6;

    padding: 20px 0 5px 0;
}


/* =========================================================
   MOBILE
   ========================================================= */

@media (max-width: 800px) {

    .gradio-container {
        padding: 10px !important;
    }

    .header {
        padding: 22px;
    }

    .header h1 {
        font-size: 25px;
    }

    .sidebar {
        margin-bottom: 15px;
    }
}
"""


# ============================================================
# HEADER
# ============================================================

header = """

<div class="header">

    <h1>
        🎓 Internship Support Assistant
    </h1>

    <p>
        Your AI-powered assistant for internship questions,
        guidance and support.
    </p>

    <div class="online">
        🟢 AI Assistant Online
    </div>

</div>

"""


# ============================================================
# SIDEBAR
# ============================================================

sidebar = """

<div class="sidebar">

    <div class="sidebar-title">
        📚 Support Categories
    </div>

    <div class="sidebar-subtitle">
        Ask your question about any internship topic.
    </div>

    <div class="category">
        📄 Reports & Documentation
    </div>

    <div class="category">
        👨‍💼 Supervisor Support
    </div>

    <div class="category">
        🕐 Working Hours
    </div>

    <div class="category">
        📅 Attendance & Leave
    </div>

    <div class="category">
        🎓 Certificates
    </div>

    <div class="category">
        📊 Performance Evaluation
    </div>

    <div class="category">
        🗂️ Required Documents
    </div>

    <div class="category">
        🏢 HR Support
    </div>

    <div class="category">
        💰 Stipend Information
    </div>

</div>

"""


# ============================================================
# CHAT RESPONSE
# ============================================================

def respond(message, history):

    if not message or message.strip() == "":
        return "", history

    answer = final_chatbot(message)

    history.append(
        (message, answer)
    )

    return "", history


# ============================================================
# BUILD APPLICATION
# ============================================================

with gr.Blocks(
    title="Internship Support Assistant"
) as demo:

    # HEADER
    gr.HTML(header)


    # MAIN CONTENT
    with gr.Row():

        # -------------------------
        # LEFT SIDEBAR
        # -------------------------

        with gr.Column(
            scale=1,
            min_width=230
        ):

            gr.HTML(sidebar)


        # -------------------------
        # RIGHT CHAT
        # -------------------------

        with gr.Column(
            scale=3
        ):

            gr.Markdown(
                """
### 💬 Chat with your Internship Assistant

Ask questions about your internship, reports,
attendance, working hours, supervisors, leave,
certificates, HR, stipend and more.
"""
            )


            chatbot = gr.Chatbot(
                label="Conversation",
                height=500
            )


            # INPUT + SEND
            with gr.Row():

                message = gr.Textbox(
                    placeholder="Type your internship question here...",
                    show_label=False,
                    lines=2,
                    scale=5
                )

                send = gr.Button(
                    "Send ➤",
                    scale=1,
                    elem_classes="primary-btn"
                )


            # CLEAR BUTTON
            clear = gr.Button(
                "🗑️ Clear Conversation",
                elem_classes="secondary-btn"
            )


            # EXAMPLES
            gr.Markdown(
                "### 💡 Example Questions"
            )

            gr.Examples(
                examples=[
                    "How do I submit my internship report?",
                    "What are the working hours?",
                    "How can I contact my supervisor?",
                    "How many days of leave can I take?",
                    "When will I receive my internship certificate?",
                    "How is my internship performance evaluated?",
                    "What documents are required?",
                    "How can I contact HR?",
                    "When will I receive my stipend?"
                ],
                inputs=message
            )


    # ========================================================
    # FOOTER
    # ========================================================

    gr.HTML(
        """
        <div class="footer">

            Internship Support Assistant
            • AI-Powered NLP System

            <br>

            Designed for Intern Support & Guidance

        </div>
        """
    )


    # ========================================================
    # EVENTS
    # ========================================================

    send.click(
        respond,
        inputs=[message, chatbot],
        outputs=[message, chatbot]
    )

    message.submit(
        respond,
        inputs=[message, chatbot],
        outputs=[message, chatbot]
    )

    clear.click(
        lambda: [],
        outputs=chatbot
    )


# ============================================================
# LAUNCH
# ============================================================

demo.launch(
    share=True,
    css=custom_css
)

predicted_categories = []

for question in test_questions:

    processed = preprocess_text(question)
    vector = loaded_vectorizer.transform([processed])

    similarities = cosine_similarity(
        vector,
        loaded_X
    )[0]

    best_index = np.argmax(similarities)

    predicted_category = loaded_df.iloc[
        best_index
    ]["category"]

    predicted_categories.append(predicted_category)

results = pd.DataFrame({
    "Question": test_questions,
    "Expected": expected_categories,
    "Predicted": predicted_categories
})

results

from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    expected_categories,
    predicted_categories
)

print(f"Chatbot Category Accuracy: {accuracy * 100:.2f}%")

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

labels = sorted(
    list(set(expected_categories + predicted_categories))
)

cm = confusion_matrix(
    expected_categories,
    predicted_categories,
    labels=labels
)

# Create the figure size BEFORE plotting
fig, ax = plt.subplots(figsize=(12, 8))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=labels
)

disp.plot(
    xticks_rotation=90,
    ax=ax
)

plt.title("Internship Chatbot Confusion Matrix")
plt.tight_layout()
plt.show()

requirements = """pandas
numpy
scikit-learn
nltk
joblib
gradio
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt created.")

